# Metodología  
1. hacer reducción dimensional con regularización lasso o con Randon Forest
2. 

In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
from scipy import stats
from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.model_selection import TimeSeriesSplit
import optuna
from sklearn.feature_selection import SelectFromModel

# ==================== CONFIGURACIÓN ====================
input_file = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\1_raw\1_meteo_epi_2021-2026_1_rezagos_sin_nulos.xlsx"
output_dir = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados"
processed_dir = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\2_procesados"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

# ==================== CARGA Y PREPROCESAMIENTO ====================
print("Cargando datos...")
df = pd.read_excel(input_file)
df['fecha'] = pd.to_datetime(df['fecha'])

# Identificar columnas
target_col = 'casos_dengue'
exclude_cols = ['fecha', 'año', 'semana_epi']
predictor_cols = [col for col in df.columns if col not in exclude_cols + [target_col]]

print(f"Predictores iniciales: {len(predictor_cols)}")

# ==================== INGENIERÍA DE ATRIBUTOS AVANZADA ====================
print("\nAplicando ingeniería de atributos avanzada...")

# Crear nuevas características basadas en tendencia y estacionalidad
df_engineered = df.copy()

# 1. Características de tendencia
df_engineered['tendencia'] = df_engineered[target_col].rolling(window=13, min_periods=1).mean()
df_engineered['tendencia_lag1'] = df_engineered['tendencia'].shift(1)
df_engineered['tendencia_lag2'] = df_engineered['tendencia'].shift(2)
df_engineered['tendencia_lag3'] = df_engineered['tendencia'].shift(3)

# 2. Características de estacionalidad
for lag in [4, 8, 12, 16, 20, 24]:
    df_engineered[f'estacionalidad_lag_{lag}'] = df_engineered[target_col].shift(lag)
    df_engineered[f'estacionalidad_diff_lag_{lag}'] = df_engineered[target_col] - df_engineered[target_col].shift(lag)

# 3. Características de cambio y aceleración
df_engineered['diff_1'] = df_engineered[target_col].diff(1)
df_engineered['diff_2'] = df_engineered[target_col].diff(2)
df_engineered['diff_3'] = df_engineered[target_col].diff(3)
df_engineered['acceleration'] = df_engineered['diff_1'].diff(1)

# 4. Características de ratio y cambio porcentual
df_engineered['pct_change_1'] = df_engineered[target_col].pct_change(1) * 100
df_engineered['pct_change_2'] = df_engineered[target_col].pct_change(2) * 100
df_engineered['ratio_lag1'] = df_engineered[target_col] / (df_engineered[target_col].shift(1) + 1)

# 5. Características de ventanas móviles
for window in [3, 5, 7, 13, 26]:
    df_engineered[f'roll_mean_{window}'] = df_engineered[target_col].rolling(window=window, min_periods=1).mean()
    df_engineered[f'roll_std_{window}'] = df_engineered[target_col].rolling(window=window, min_periods=1).std()
    df_engineered[f'roll_max_{window}'] = df_engineered[target_col].rolling(window=window, min_periods=1).max()
    df_engineered[f'roll_min_{window}'] = df_engineered[target_col].rolling(window=window, min_periods=1).min()

# 6. Características de rango
df_engineered['range_7'] = df_engineered[f'roll_max_7'] - df_engineered[f'roll_min_7']
df_engineered['range_13'] = df_engineered[f'roll_max_13'] - df_engineered[f'roll_min_13']

# 7. Características de picos (detección de anomalías)
z_scores = np.abs(stats.zscore(df_engineered[target_col].fillna(0)))
df_engineered['es_pico'] = (z_scores > 2).astype(int)
df_engineered['es_pico_lag1'] = df_engineered['es_pico'].shift(1)
df_engineered['es_pico_lag2'] = df_engineered['es_pico'].shift(2)

# 8. Características de interacción con variables meteorológicas
meteo_vars = [col for col in predictor_cols if col.startswith(('prec', 'temp', 'tmax', 'tmin', 'hr', 'soi', 'oni', 'mei'))]
for var in meteo_vars[:5]:  # Limitar para no crear demasiadas
    df_engineered[f'{var}_x_casos_lag1'] = df_engineered[var] * df_engineered[target_col].shift(1)
    df_engineered[f'{var}_x_tendencia'] = df_engineered[var] * df_engineered['tendencia']

# Limpiar el dataframe (eliminar NAs)
df_engineered = df_engineered.dropna()

# Actualizar lista de predictores
new_predictor_cols = [col for col in df_engineered.columns if col not in exclude_cols + [target_col]]
print(f"Predictores después de ingeniería: {len(new_predictor_cols)}")

# ==================== SELECCIÓN DE CARACTERÍSTICAS CON RANDOM FOREST ====================
print("\nSeleccionando características con Random Forest...")

# Preparar datos para RF
X_rf = df_engineered[new_predictor_cols].values
y_rf = df_engineered[target_col].values

# Entrenar Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_rf, y_rf)

# Obtener importancia de características
feature_importance = pd.DataFrame({
    'Feature': new_predictor_cols,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

# Seleccionar top N features (mantener < 30 para LightGBM)
n_selected = min(30, len(new_predictor_cols))
selected_features_rf = feature_importance.head(n_selected)['Feature'].tolist()

print(f"Top {n_selected} características seleccionadas por RF:")
for i, (feature, imp) in enumerate(zip(selected_features_rf[:10], feature_importance['Importance'][:10])):
    print(f"  {i+1}. {feature}: {imp:.4f}")

# Guardar lista de atributos seleccionados en Excel
features_file = os.path.join(output_dir, 'atributos_seleccionados_rf.xlsx')
feature_importance.to_excel(features_file, index=False)
print(f"\nLista de atributos seleccionados guardada en: {features_file}")

# ==================== PREPARACIÓN DE DATOS FINALES ====================
print("\nPreparando datos finales...")

# Usar solo features seleccionados
X = df_engineered[selected_features_rf].values
y = df_engineered[target_col].values
fechas = df_engineered['fecha'].values
años = df_engineered['año'].values
semanas = df_engineered['semana_epi'].values

# Dividir en entrenamiento y test
train_mask = (años >= 2021) & (años <= 2025)
test_mask = años == 2026

X_train = X[train_mask]
y_train = y[train_mask]
X_test = X[test_mask]
y_test = y[test_mask]
fechas_train = fechas[train_mask]
fechas_test = fechas[test_mask]
semanas_train = semanas[train_mask]
semanas_test = semanas[test_mask]

print(f"Entrenamiento: {len(X_train)} registros")
print(f"Test: {len(X_test)} registros")

# ==================== OPTIMIZACIÓN Y ENTRENAMIENTO AVANZADO ====================
print("\nOptimizando modelo LightGBM con enfoque en picos...")

# Dividir para validación
val_mask = (años >= 2021) & (años <= 2024)
opt_train_mask = años == 2025

X_opt_train = X[val_mask]
y_opt_train = y[val_mask]
X_opt_val = X[opt_train_mask]
y_opt_val = y[opt_train_mask]

def create_advanced_weights(y, peak_threshold=0.80, extreme_threshold=0.95, 
                           trend_weight=1.0, increase_weight=1.3):
    """
    Crea pesos avanzados considerando magnitud, tendencia e incrementos
    """
    weights = np.ones_like(y, dtype=float)
    
    # Pesos por magnitud
    threshold_high = np.percentile(y, peak_threshold * 100)
    threshold_extreme = np.percentile(y, extreme_threshold * 100)
    
    # Pesos exponenciales para picos más altos
    for i, val in enumerate(y):
        if val >= threshold_extreme:
            # Peso exponencial para valores extremos
            weight_factor = 3.0 + (val - threshold_extreme) / (threshold_extreme + 1) * 2.0
            weights[i] *= weight_factor
        elif val >= threshold_high:
            # Peso lineal para valores altos
            weight_factor = 1.5 + (val - threshold_high) / (threshold_high + 1) * 1.5
            weights[i] *= weight_factor
    
    # Peso adicional para semanas con tendencia ascendente fuerte
    if len(y) > 3:
        for i in range(3, len(y)):
            if y[i] > y[i-1] and y[i] > y[i-2] and y[i] > y[i-3]:
                weights[i] *= 1.3
    
    # Peso adicional para semanas con incremento significativo
    if len(y) > 1:
        diff = np.diff(y)
        diff_high = np.percentile(np.abs(diff), 80)
        for i in range(1, len(y)):
            if diff[i-1] > diff_high:
                weights[i] *= 1.2
    
    return weights

def optimize_for_peaks_advanced(X_train, y_train, X_val, y_val):
    """
    Optimización avanzada para picos
    """
    def objective(trial):
        params = {
            'objective': 'regression',
            'metric': 'mae',
            'boosting_type': 'gbdt',
            'num_leaves': trial.suggest_int('num_leaves', 20, 80),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 2.0),
            'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.5),
            'max_depth': trial.suggest_int('max_depth', 5, 12),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'verbose': -1,
            'n_jobs': -1,
            'random_state': 42
        }
        
        # Crear pesos avanzados
        val_weights = create_advanced_weights(y_val)
        
        dtrain = lgb.Dataset(X_train, y_train)
        dval = lgb.Dataset(X_val, y_val, weight=val_weights, reference=dtrain)
        
        model = lgb.train(
            params,
            dtrain,
            valid_sets=[dtrain, dval],
            valid_names=['train', 'val'],
            num_boost_round=3000,
            callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)]
        )
        
        y_pred = model.predict(X_val, num_iteration=model.best_iteration)
        
        # Métrica compuesta: 70% MAE general + 30% MAE en picos
        general_mae = mean_absolute_error(y_val, y_pred)
        
        peak_mask = y_val > np.percentile(y_val, 80)
        if np.sum(peak_mask) > 0:
            peak_mae = mean_absolute_error(y_val[peak_mask], y_pred[peak_mask])
        else:
            peak_mae = general_mae
        
        # Penalizar subestimación severa
        severe_under = np.mean(np.maximum(0, y_val[peak_mask] - y_pred[peak_mask])) if np.sum(peak_mask) > 0 else 0
        
        # Score combinado
        combined_score = 0.7 * general_mae + 0.3 * peak_mae + 0.1 * severe_under
        
        return combined_score
    
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    
    print(f"\nMejores parámetros encontrados:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    return study.best_params

# Optimizar
best_params = optimize_for_peaks_advanced(X_opt_train, y_opt_train, X_opt_val, y_opt_val)

# Crear pesos avanzados para entrenamiento completo
train_weights = create_advanced_weights(y_train)

# Parámetros base
base_params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'verbose': -1,
    'n_jobs': -1,
    'random_state': 42
}

# Entrenar modelo final
model_params = {**base_params, **best_params}
dtrain_final = lgb.Dataset(X_train, y_train, weight=train_weights)
dtest_final = lgb.Dataset(X_test, y_test, reference=dtrain_final)

print("\nEntrenando modelo final...")
model = lgb.train(
    model_params,
    dtrain_final,
    valid_sets=[dtrain_final, dtest_final],
    valid_names=['train', 'test'],
    num_boost_round=5000,
    callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)]
)

# Predicciones
y_train_pred = model.predict(X_train, num_iteration=model.best_iteration)
y_test_pred = model.predict(X_test, num_iteration=model.best_iteration)

# ==================== MÉTRICAS ====================
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

# Métricas en picos
peak_mask_train = y_train > np.percentile(y_train, 80)
peak_mask_test = y_test > np.percentile(y_test, 80)
peak_mae_train = mean_absolute_error(y_train[peak_mask_train], y_train_pred[peak_mask_train]) if np.sum(peak_mask_train) > 0 else np.nan
peak_mae_test = mean_absolute_error(y_test[peak_mask_test], y_test_pred[peak_mask_test]) if np.sum(peak_mask_test) > 0 else np.nan

print("\n" + "="*60)
print("RESULTADOS DEL MODELO MEJORADO")
print("="*60)
print(f"MAE Train: {train_mae:.2f}")
print(f"MAE Test: {test_mae:.2f}")
print(f"Peak MAE Train: {peak_mae_train:.2f}")
print(f"Peak MAE Test: {peak_mae_test:.2f}")
print(f"R² Train: {train_r2:.4f}")
print(f"R² Test: {test_r2:.4f}")
print("="*60)

# ==================== GRÁFICOS COMPARATIVOS ====================
print("\nGenerando gráficos...")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Comparativa Modelo LightGBM Mejorado - Ingeniería de Atributos + RF', fontsize=14, fontweight='bold')

# Gráfico 1: Entrenamiento
ax1.plot(fechas_train, y_train, 'b-', label='Real', linewidth=1.5, alpha=0.7)
ax1.plot(fechas_train, y_train_pred, 'r-', label='Predicho', linewidth=1.5, alpha=0.7)
# Resaltar picos
peak_mask_train_plot = y_train > np.percentile(y_train, 80)
ax1.scatter(fechas_train[peak_mask_train_plot], y_train[peak_mask_train_plot], 
           color='gold', s=30, label='Picos (>80%)', zorder=5, alpha=0.8)
ax1.set_xlabel('Fecha')
ax1.set_ylabel('Casos Dengue')
ax1.set_title(f'Entrenamiento (2021-2025) - MAE: {train_mae:.2f}')
ax1.legend(loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Gráfico 2: Test
ax2.plot(fechas_test, y_test, 'b-', label='Real', linewidth=1.5, alpha=0.7)
ax2.plot(fechas_test, y_test_pred, 'r-', label='Predicho', linewidth=1.5, alpha=0.7)
# Resaltar picos
peak_mask_test_plot = y_test > np.percentile(y_test, 80)
ax2.scatter(fechas_test[peak_mask_test_plot], y_test[peak_mask_test_plot], 
           color='gold', s=30, label='Picos (>80%)', zorder=5, alpha=0.8)
ax2.set_xlabel('Fecha')
ax2.set_ylabel('Casos Dengue')
ax2.set_title(f'Test (2026) - MAE: {test_mae:.2f}')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()

# Guardar gráfico
plot_file = os.path.join(output_dir, 'comparativa_entrenamiento_test_mejorado.png')
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
print(f"Gráfico guardado en: {plot_file}")
plt.close()

# ==================== GUARDAR DATASET PROCESADO ====================
print("\nGuardando dataset procesado...")
processed_file = os.path.join(processed_dir, 'dataset_procesado_ingenieria_rf.xlsx')

# Guardar con todas las columnas relevantes
df_final = df_engineered.copy()
df_final = df_final[exclude_cols + [target_col] + selected_features_rf]
df_final.to_excel(processed_file, index=False)
print(f"Dataset procesado guardado en: {processed_file}")

# ==================== RESULTADOS EN EXCEL ====================
print("\nGenerando Excel con resultados detallados...")
excel_file = os.path.join(output_dir, 'resultados_modelo_mejorado.xlsx')

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Métricas
    metrics_df = pd.DataFrame({
        'Métrica': ['MAE', 'RMSE', 'R²', 'Peak MAE (80%)'],
        'Entrenamiento': [train_mae, train_rmse, train_r2, peak_mae_train],
        'Test': [test_mae, test_rmse, test_r2, peak_mae_test]
    })
    metrics_df.to_excel(writer, sheet_name='Métricas', index=False)
    
    # Predicciones
    pred_df = pd.DataFrame({
        'fecha': fechas_test,
        'año': años[test_mask],
        'semana_epi': semanas_test,
        'casos_reales': y_test,
        'predicciones': y_test_pred,
        'error': np.abs(y_test - y_test_pred),
        'es_pico': y_test > np.percentile(y_test, 80)
    })
    pred_df.to_excel(writer, sheet_name='Predicciones_Test', index=False)
    
    # Características seleccionadas
    feature_importance_df = pd.DataFrame({
        'Feature': selected_features_rf,
        'Importance': rf.feature_importances_[:len(selected_features_rf)]
    }).sort_values('Importance', ascending=False)
    feature_importance_df.to_excel(writer, sheet_name='Features_RF', index=False)
    
    # Parámetros del modelo
    params_df = pd.DataFrame({
        'Parámetro': list(best_params.keys()),
        'Valor': [str(v) for v in best_params.values()]
    })
    params_df.to_excel(writer, sheet_name='Parámetros', index=False)
    
    # Análisis de errores por rango
    bins = [0, 5, 10, 20, 50, 100, 200]
    labels = ['0-5', '5-10', '10-20', '20-50', '50-100', '100-200']
    pred_df['rango_casos'] = pd.cut(pred_df['casos_reales'], bins=bins, labels=labels)
    
    error_analysis = pd.DataFrame()
    for label in labels:
        mask = pred_df['rango_casos'] == label
        if mask.sum() > 0:
            error_analysis.loc[label, 'Count'] = mask.sum()
            error_analysis.loc[label, 'MAE'] = pred_df[mask]['error'].mean()
            error_analysis.loc[label, 'Max_Error'] = pred_df[mask]['error'].max()
    error_analysis.to_excel(writer, sheet_name='Análisis_Errores')

print(f"Excel guardado en: {excel_file}")

# ==================== GUARDAR MODELO ====================
model_file = os.path.join(output_dir, 'modelo_mejorado_final.txt')
model.save_model(model_file)
print(f"Modelo guardado en: {model_file}")

# ==================== RESUMEN FINAL ====================
print("\n" + "="*70)
print("RESUMEN FINAL - MODELO MEJORADO")
print("="*70)
print(f"✓ Características iniciales: {len(predictor_cols)}")
print(f"✓ Características después de ingeniería: {len(new_predictor_cols)}")
print(f"✓ Características seleccionadas por RF: {len(selected_features_rf)}")
print(f"\n✓ MAE Entrenamiento: {train_mae:.2f}")
print(f"✓ MAE Test: {test_mae:.2f}")
print(f"✓ MAE Picos (Test): {peak_mae_test:.2f}")
print(f"✓ R² Test: {test_r2:.4f}")

# Mejora vs modelo anterior
improvement = ((8.70 - test_mae) / 8.70) * 100
peak_improvement = ((12.65 - peak_mae_test) / 12.65) * 100 if not np.isnan(peak_mae_test) else 0

print(f"\n✓ Mejora en MAE General: {improvement:.1f}%")
print(f"  (Anterior: 8.70 → Nuevo: {test_mae:.2f})")
print(f"\n✓ Mejora en MAE de Picos: {peak_improvement:.1f}%")
print(f"  (Anterior: 12.65 → Nuevo: {peak_mae_test:.2f})")

print("\nEstrategias implementadas:")
print("1. Ingeniería de atributos avanzada (tendencia, estacionalidad, ventanas)")
print("2. Selección de características con Random Forest")
print("3. Pesos avanzados para picos (exponenciales y por tendencia)")
print("4. Optimización con Optuna (función objetivo combinada)")
print("5. Feature engineering enfocado en detección de picos")

print("\nArchivos generados:")
print(f"  - Dataset procesado: {processed_file}")
print(f"  - Features seleccionados: {features_file}")
print(f"  - Gráfico comparativo: {plot_file}")
print(f"  - Resultados Excel: {excel_file}")
print(f"  - Modelo guardado: {model_file}")
print("="*70)


Cargando datos...
Predictores iniciales: 168

Aplicando ingeniería de atributos avanzada...
Predictores después de ingeniería: 226

Seleccionando características con Random Forest...


[I 2026-07-31 11:17:30,171] A new study created in memory with name: no-name-134ad6c0-e105-4b50-baf2-1092eea6f762


Top 30 características seleccionadas por RF:
  1. roll_mean_3: 0.6371
  2. roll_max_3: 0.1097
  3. roll_min_3: 0.0903
  4. es_pico: 0.0781
  5. roll_max_5: 0.0165
  6. pct_change_2: 0.0051
  7. diff_1: 0.0048
  8. diff_2: 0.0040
  9. ratio_lag1: 0.0032
  10. diff_3: 0.0031

Lista de atributos seleccionados guardada en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados\atributos_seleccionados_rf.xlsx

Preparando datos finales...
Entrenamiento: 225 registros
Test: 21 registros

Optimizando modelo LightGBM con enfoque en picos...


  0%|          | 0/50 [00:00<?, ?it/s]

Training until validation scores don't improve for 150 rounds


...

Best trial: 12. Best value: 7.66576:  98%|█████████▊| 49/50 [00:18<00:00,  2.70it/s]

Early stopping, best iteration is:
[443]	train's l1: 0.226699	val's l1: 7.70089
[I 2026-07-31 11:17:49,155] Trial 48 finished with value: 8.54802125964175 and parameters: {'num_leaves': 59, 'learning_rate': 0.03228617739486484, 'feature_fraction': 0.903629508781975, 'bagging_fraction': 0.9361974442750775, 'bagging_freq': 1, 'min_child_samples': 5, 'reg_alpha': 1.4543474868371158, 'reg_lambda': 0.36097203857332716, 'min_split_gain': 0.22965454840835342, 'max_depth': 11, 'subsample': 0.9533086016669818, 'colsample_bytree': 0.9304040422196903}. Best is trial 12 with value: 7.665756369310081.
Training until validation scores don't improve for 150 rounds


Best trial: 12. Best value: 7.66576: 100%|██████████| 50/50 [00:19<00:00,  2.52it/s]

Early stopping, best iteration is:
[2763]	train's l1: 0.304908	val's l1: 10.3272
[I 2026-07-31 11:17:50,037] Trial 49 finished with value: 12.399688628987102 and parameters: {'num_leaves': 47, 'learning_rate': 0.00513790183937535, 'feature_fraction': 0.9995121190073148, 'bagging_fraction': 0.9388083237870766, 'bagging_freq': 2, 'min_child_samples': 8, 'reg_alpha': 1.2651393226574847, 'reg_lambda': 0.36174274043448174, 'min_split_gain': 0.29727376142605455, 'max_depth': 11, 'subsample': 0.9235232597915154, 'colsample_bytree': 0.9318600388960646}. Best is trial 12 with value: 7.665756369310081.

Mejores parámetros encontrados:
  num_leaves: 79
  learning_rate: 0.028326596637011918
  feature_fraction: 0.8037076130857927
  bagging_fraction: 0.9238465032542403
  bagging_freq: 7
  min_child_samples: 5
  reg_alpha: 1.95064752388159
  reg_lambda: 0.006445114477759417
  min_split_gain: 0.204503894400332
  max_depth: 11
  subsample: 0.8130026348020827
  colsample_bytree: 0.6186126500384421

Entr

Early stopping, best iteration is:
[322]	train's l1: 0.2228	test's l1: 2.65511

RESULTADOS DEL MODELO MEJORADO
MAE Train: 0.23
MAE Test: 2.66
Peak MAE Train: 0.21
Peak MAE Test: 4.66
R² Train: 0.9998
R² Test: 0.8516

Generando gráficos...
Gráfico guardado en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados\comparativa_entrenamiento_test_mejorado.png

Guardando dataset procesado...
Dataset procesado guardado en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\2_datos\2_procesados\dataset_procesado_ingenieria_rf.xlsx

Generando Excel con resultados detallados...
Excel guardado en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados\resultados_modelo_mejorado.xlsx
Modelo guardado en: C:\Users\marco\Documentos\investigacion\machine_learning_idalina\4_algoritmo_LightGBM\3_resultados\modelo_mejorado_final.txt

RESUMEN FINAL - MODELO MEJORADO
✓ Característica

**Mejoras clave implementadas:**

1. **Ingeniería de atributos avanzada**:
   - Variables de tendencia (media móvil, diferencias)
   - Variables de estacionalidad (lags estacionales)
   - Ventanas móviles (media, desviación, máximo, mínimo)
   - Detección de picos (z-score)
   - Interacciones con variables meteorológicas

2. **Selección de características con Random Forest**:
   - Selecciona automáticamente las 30 características más importantes
   - Guarda lista detallada en Excel
   - Reduce ruido y mejora interpretabilidad

3. **Pesos avanzados**:
   - Pesos exponenciales para picos más altos
   - Peso adicional por tendencia ascendente
   - Peso adicional por incrementos significativos

4. **Optimización combinada**:
   - Función objetivo que balancea MAE general y MAE en picos
   - Penaliza subestimación severa
   - Validación con datos de 2025

5. **Visualización mejorada**:
   - Gráficos lado a lado (entrenamiento vs test)
   - MAE en títulos de cada gráfico
   - Picos resaltados en ambos conjuntos

6. **Resultados completos**:
   - Dataset procesado guardado
   - Lista de atributos seleccionados
   - Análisis de errores por rango de casos
   - Métricas detalladas en Excel

**Resultados esperados:**
- MAE Test < 7.0
- MAE en picos significativamente reducido
- Mejor captura de picos en gráficos
- Modelo más interpretable con menos características